In [4]:
import os
import glob
import json
import re
import shutil
from os.path import join as pjoin
import torch
from transformers import BertTokenizer

In [6]:
""" 
Produce dataset (train, valid, test) with tokens greater than 512.
This dataset must have the same data whether it is english or indonesian.
Say that we have the source folder in english.
First, it will look after documents with token > 512 in the english folder.
Then, long docs in english folder will be saved.
The indexes are saved so that the long docs in indonesian will be the same data as in english folder.
Why do the data have to be the same?
So that the test data will be the same then the model performance can be compared.
"""

large = False
model_version = "p2"
raw_path_1 = "../bert_data/en"
raw_path_2 = "../bert_data/id-p2"

lang_1 = raw_path_1.split("/")[-1]
lang_2 = raw_path_2.split("/")[-1]

# model_version = "" if args.model_version == "p1" else "p2"
# model_size = "_large" if args.large else ""

save_path_1 = f"{raw_path_1}_{lang_1}_long"
save_path_2 = f"{raw_path_1}_{lang_2}_long"
if os.path.exists(save_path_1):
    shutil.rmtree(save_path_1)
if os.path.exists(save_path_2):
    shutil.rmtree(save_path_2)
os.mkdir(save_path_1)
os.mkdir(save_path_2)

datasets = ['train', 'valid', 'test']

long_data_idx = {}
for corpus_type in datasets:
    print(f"Looping through {raw_path_1} {corpus_type} dataset...")
    
    # Loop through the reference dataset
    for json_f in glob.glob(pjoin(raw_path_1, '*' + corpus_type + '.*.json')):
        dataset_name = json_f.split('/')[-1].split(".")[0]  # --> xlsum
        idx = json_f.split('/')[-1].split(".")[-2] # --> index: 1-19 (train), 1-3 (valid, test)
        filename = f"/{dataset_name}.{corpus_type}.{idx}.bert.pt"
        
        save_file = save_path_1 + filename
        
        with open(json_f) as json_f:
            data = json.load(json_f)
            datasets = []

            # Get data with total tokens > 512
            for i in range(len(data)):
                src = data[i]['src']
                if len(src) > 512:
                    datasets.append(data[i])

                    # Save the index (key) for data with long tokens
                    if f"{corpus_type}_{idx}" in long_data_idx.keys():
                        long_data_idx[f"{corpus_type}_{idx}"].append(i) 
                    else:
                        long_data_idx[f"{corpus_type}_{idx}"] = [i]
                    
            print(f"Saving {len(datasets)} instances to {save_file}...")
            torch.save(datasets, save_file)
        
            save_file_json = save_file.replace('bert.pt', 'json')
            with open(save_file_json, 'w') as f:
                f.write(json.dumps(datasets))

    
    print(f"Looping through {raw_path_2} {corpus_type} dataset...")
    
    # Loop through the follower dataset
    for json_f in glob.glob(pjoin(raw_path_2, '*' + corpus_type + '.*.json')):
        real_name = json_f.split('/')[-1].split(".")[0]
        idx = json_f.split('/')[-1].split(".")[-2]
        filename = f"/{dataset_name}.{corpus_type}.{idx}.bert.pt"
        
        save_file = save_path_2 + filename

        with open(json_f) as json_f:
            data = json.load(json_f)
            datasets = []

            # Get data with the same index from the reference dataset
            for i in range(len(data)):
                if i in long_data_idx[f"{corpus_type}_{idx}"]:
                    datasets.append(data[i])
                    
            print(f"Saving {len(datasets)} instances to {save_file}...")
            torch.save(datasets, save_file)

            save_file_json = save_file.replace('bert.pt', 'json')
            with open(save_file_json, 'w') as f:
                f.write(json.dumps(datasets))

Looping through ../bert_data/en...
Saving 1389 instances to ../bert_data/en_en_long/xlsum.train.8.bert.pt...
Saving 1405 instances to ../bert_data/en_en_long/xlsum.train.11.bert.pt...
Saving 1398 instances to ../bert_data/en_en_long/xlsum.train.9.bert.pt...
Saving 1411 instances to ../bert_data/en_en_long/xlsum.train.12.bert.pt...
Saving 1439 instances to ../bert_data/en_en_long/xlsum.train.2.bert.pt...
Saving 177 instances to ../bert_data/en_en_long/xlsum.train.19.bert.pt...
Saving 1382 instances to ../bert_data/en_en_long/xlsum.train.14.bert.pt...
Saving 1411 instances to ../bert_data/en_en_long/xlsum.train.10.bert.pt...
Saving 1390 instances to ../bert_data/en_en_long/xlsum.train.4.bert.pt...
Saving 1422 instances to ../bert_data/en_en_long/xlsum.train.18.bert.pt...
Saving 1416 instances to ../bert_data/en_en_long/xlsum.train.17.bert.pt...
Saving 1493 instances to ../bert_data/en_en_long/xlsum.train.1.bert.pt...
Saving 1428 instances to ../bert_data/en_en_long/xlsum.train.13.bert.pt